In [6]:
%cd ..
%load_ext autoreload
%autoreload 2

# Configure logger to ignore everything to avoid cluttering the output
import logging
logging.getLogger().setLevel(logging.WARNING)

import dotenv # load env vars from .env
dotenv.load_dotenv()

from openai import OpenAI
import dotenv  
import os   

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
%%capture
from src.examples.agent.design_w_promoter_vars import generate_chat_histories   

generate_chat_histories(output_dir="outputs/chat_histories", num_attempts=100, start_index=0)

In [2]:
from src.examples.agent.design_w_promoter_vars import scores_for_runs_from_directory

MODEL_NAME = "gpt-4.1-2025-04-14" # chat histories are stored under a model specific directory
scores = scores_for_runs_from_directory(f"outputs/chat_histories/{MODEL_NAME}")

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Here are the values we hope to improve after training

score_values = [scores[k]['score'] for k in scores.keys()]
tool_failure_values = [scores[k]['num_tool_failures'] for k in scores.keys()]
print(f"Mean score: {np.mean(score_values)}")
print(f"Std score: {np.std(score_values)}")
print(f"Min score: {np.min(score_values)}")
print(f"Max score: {np.max(score_values)}")

num_tool_calls = sum([scores[k]['num_tool_calls'] for k in scores.keys()])
num_messages = sum([scores[k]['num_messages'] for k in scores.keys()])
num_tool_failures = sum(tool_failure_values)

print(f"Average number of messages: {num_messages / len(scores.keys())}")
print(f"Average number of tool calls: {num_tool_calls / len(scores.keys())}")
print(f"Average number of tool failures: {num_tool_failures / len(scores.keys())}")

# # show distribution of scores
plt.figure(figsize=(8, 2))
plt.hist(score_values, bins=10)
plt.show()

# show distribution of scores
plt.figure(figsize=(8, 2))
plt.hist(tool_failure_values, bins=10)
plt.show()

In [ ]:
sorted_scores = sorted(scores.keys(), key=lambda x: scores[x]['score'])

print('Best score: ', sorted_scores[0], scores[sorted_scores[0]]['score'])
print('Worst score: ', sorted_scores[-1], scores[sorted_scores[-1]]['score'])

# Direct Preference Optimization

#### Collect preferred and non-preferred response

In [ ]:
import os
import json
from src.examples.agent.design_w_promoter_vars import get_runner

output_dir = "outputs/dpo_pairs"

start_index = 0
num_attempts = 10
all_dpo_pairs = []  
for attempt_index in range(start_index, start_index + num_attempts):
    run_id = f"design_w_promoter_vars_dataset_{attempt_index}"

    runner = get_runner(max_rounds=25, max_attempts=3)
    print(f"Running {run_id}")
    dpo_pairs = runner.run_collect_dpo_pairs()
    print(f"Collected {len(dpo_pairs)} dpo pairs")
    all_dpo_pairs += dpo_pairs
    os.makedirs(f"{output_dir}/{runner.model}/{run_id}", exist_ok=True)
    with open(f"{output_dir}/{runner.model}/{run_id}/dpo_pairs.jsonl", "w") as f:
        for dpo_pair in dpo_pairs:
            f.write(json.dumps(dpo_pair) + "\n")
            
with open(f'{output_dir}/{runner.model}/design_w_promoter_vars_dataset_all_dpo_pairs.jsonl', "w") as f:
    for dpo_pair in all_dpo_pairs:
        f.write(json.dumps(dpo_pair) + "\n")

In [ ]:
### Upload the training file and run the fine-tuning job

In [ ]:
# test on a single run
dpo_training_set_path = "outputs/dpo_pairs/design_w_promoter_vars_dataset_all_dpo_pairs.jsonl"

# upload the training set to the cloud
uploaded_file = client.files.create(
    file=open(dpo_training_set_path, "rb"),
    purpose="fine-tune",
)
uploaded_file.id

job = client.fine_tuning.jobs.create(
    training_file=uploaded_file.id,
    model="gpt-4.1-2025-04-14",
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {"beta": 0.1},
        },
    },
)

print(job)

In [ ]:

# https://platform.openai.com/docs/guides/direct-preference-optimization
# The data should be formatted in JSONL format, with each line representing an example in the following structure:

# {
#   "input": {
#     "messages": [
#       {
#         "role": "user",
#         "content": "Hello, can you tell me how cold San Francisco is today?"
#       }
#     ],
#     "tools": [],
#     "parallel_tool_calls": true
#   },
#   "preferred_output": [
#     {
#       "role": "assistant",
#       "content": "Today in San Francisco, it is not quite cold as expected. Morning clouds will give away to sunshine, with a high near 68°F (20°C) and a low around 57°F (14°C)."
#     }
#   ],
#   "non_preferred_output": [
#     {
#       "role": "assistant",
#       "content": "It is not particularly cold in San Francisco today."
#     }
#   ]
# }
import json
from src.tool_registry import tool_functions
def get_assistant_messages(messages):
    messages = [message for message in messages if message['role'] == 'assistant']
    # for message in messages:
    #     # tool call 
    #     if 'tool_calls' in message:
    #         tool_calls = message['tool_calls']
    #         for tool_call in tool_calls:
    #             arguments = tool_call.get('function', {}).get('arguments', {})
    #             tool_call['function']['arguments'] = json.loads(arguments)
    return messages

def get_user_and_system_messages(messages):
    user_messages = [message for message in messages if message['role'] == 'user']
    # for message in user_messages:
        # message['tools'] = tool_functions
        # message['parallel_tool_calls'] = True
    system_messages = [message for message in messages if message['role'] == 'system']
    return system_messages + user_messages

input_messages = None
samples = []
num_pairs = 10

for good, bad in zip(sorted_scores[:num_pairs], sorted_scores[-num_pairs:]):
    with open(good, "r") as f:
        good_messages = json.load(f)
        
        if input_messages is None:
            input_messages = get_user_and_system_messages(good_messages)
        
        good_response_messages = get_assistant_messages(good_messages)
        
    with open(bad, "r") as f:
        bad_messages = json.load(f)
        bad_response_messages = get_assistant_messages(bad_messages)

    sample = {"input":
                {"messages": input_messages}, 
              "preferred_output": good_response_messages, 
              "non_preferred_output": bad_response_messages}
    
    samples.append(sample)
    
    
dpo_training_set_path = "outputs/training_sets/dpo_promoter_var_dataset_gpt-4.1-2025-04-14.jsonl"
with open(dpo_training_set_path, "w") as f:
    for sample in samples:
        f.write(json.dumps(sample) + "\n")

In [ ]:
from openai import OpenAI
import dotenv  
import os   

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

# upload the training set to the cloud
uploaded_file = client.files.create(
    file=open(dpo_training_set_path, "rb"),
    purpose="fine-tune",
)
uploaded_file.id

In [ ]:
job = client.fine_tuning.jobs.create(
    training_file=uploaded_file.id,
    model="gpt-4.1-2025-04-14",
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {"beta": 0.1},
        },
    },
)

print(job)

# Reinforcement Learning 

In [ ]:
# https://platform.openai.com/docs/guides/reinforcement-fine-tuning
# 1. Implement a grader that assigns a numeric reward to each model response.
# 2. Upload your prompt dataset and designate a validation split.
# 3. Start the fine-tune job.

#     Python graders
# This grader allows you to execute arbitrary python code to grade the model output. The grader expects a grade function to be present that takes in two arguments and outputs a float value. Any other result (exception, invalid float value, etc.) will be marked as invalid and return a 0 grade.
# {
#     "type": "python",
#     "source": "def grade(sample, item):\n    return 1.0",
#     "image_tag": "2025-05-08"
# }
# The python source code must contain a grade function that takes in exactly two arguments and returns a float value as a grade.

# from typing import Any
# def grade(sample: dict[str, Any], item: dict[str, Any]) -> float:
#     # your logic here
#     return 1.0

# If you don't want to manually put your grading function in a string, you can also load it from a Python file using importlib and inspect. For example, if your grader function is in a file named grader.py, you can do:

import importlib
import inspect

grader_module = importlib.import_module("src.rl.graders.grade_design_w_promoters")
grader = {
    "type": "python",
    "source": inspect.getsource(grader_module)
}

api_key = os.environ["OPENAI_API_KEY"]
headers = {"Authorization": f"Bearer {api_key}"}

# validate the grader
# import os
# import requests
# payload = {"grader": grader}
# response = requests.post(
#     "https://api.openai.com/v1/fine_tuning/alpha/graders/validate",
#     json=payload,
#     headers=headers
# )
# print("validate request_id:", response.headers["x-request-id"])
# print("validate response:", response.text)

# The first argument supplied to the grading function will be a dictionary populated with the model’s output during training for you to grade. output_json will only be populated if the output uses response_format.
# {
#     "choices": [...],
#     "output_text": "...",
#     "output_json": {},
#     "output_tools": [...]
# }
# The second argument supplied is a dictionary populated with input grading context. For evals, this will include keys from the data source. For fine-tuning this will include keys from each training data row.

# {
#     "reference_answer": "...",
#     "my_key": {...}
# }
# Here's a working example:

# import os
# import requests

# get the API key from environment

# run the grader with a test reference and sample
payload = {
  "grader": grader,
  "item": {
     "session_state_history": session_state_history
  }
}
response = requests.post(
    "https://api.openai.com/v1/fine_tuning/alpha/graders/run",
    json=payload,
    headers=headers
)
print("run request_id:", response.headers["x-request-id"])
print("run response:", response.text)

In [ ]:
response.json()

In [ ]:

session_paths = sorted_scores[:3]
samples = []
for session_path in session_paths:
    chat_history = session_path
    session_state_history = session_path.replace("chat_history.json", "session_state.json")
    with open(chat_history, "r") as f:
        chat_history = json.load(f)
    with open(session_state_history, "r") as f:
        session_state_history = json.load(f)    
    sample = { 
        "input": {
            "messages": chat_history
        },
        "session_state_history": session_state_history
    }
    
    samples.append(sample)
    
rl_training_set_path = "outputs/training_sets/rl/promoter_var_dataset_gpt-4.1-2025-04-14.jsonl"
from pathlib import Path
Path(rl_training_set_path).parent.mkdir(parents=True, exist_ok=True)
with open(rl_training_set_path, "w") as f:
    for sample in samples:
        f.write(json.dumps(sample) + "\n")

In [ ]:
from openai import OpenAI
import dotenv  
import os   
import json

dotenv.load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

# upload the training set to the cloud
uploaded_file = client.files.create(
    file=open(rl_training_set_path, "rb"),
    purpose="fine-tune",
)
uploaded_file.id

job = client.fine_tuning.jobs.create(
    training_file=uploaded_file.filename,
    model="gpt-4.1-2025-04-14",
    method={
        "type": "reinforcement-learning",
        "rl": {
            "hyperparameters": {"beta": 0.1},
        },
    },
)

print(job)